In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import random

# Жесткая фиксация seed для воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Определение устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


In [3]:
# Базовые трансформации (без аугментаций для эксперимента C1)
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Загрузка полного тренировочного датасета
full_train_dataset = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=base_transform)

# Загрузка тестового датасета
test_dataset = torchvision.datasets.STL10(root='./data', split='test', download=True, transform=base_transform)

# Разбиение full_train на train и val (80/20)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

# Используем generator с нашим seed для предсказуемого разбиения
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

# Создание DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Sanity-check: проверяем размерности
for images, labels in train_loader:
    print(f"Размер батча изображений: {images.shape}")
    print(f"Размер батча меток: {labels.shape}")
    break

100.0%


Размер батча изображений: torch.Size([64, 3, 96, 96])
Размер батча меток: torch.Size([64])


In [4]:
import torch.nn as nn
import torch.optim as optim

# 1. Архитектура простой CNN для эксперимента C1
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 12 * 12, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(-1, 64 * 12 * 12)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model_c1 = SimpleCNN().to(device)

# 2. Инициализация функции потерь и оптимизатора
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_c1.parameters(), lr=0.001)

# 3. Реализация базовых функций обучения и оценки
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# 4. Запуск цикла обучения
epochs = 5 # Для начала используем 5 эпох, чтобы проверить пайплайн
c1_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_c1 = 0.0

print("Запуск эксперимента C1 (simple-cnn-base)...")
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model_c1, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model_c1, val_loader, criterion, device)
    
    # Логирование истории обучения
    c1_history['train_loss'].append(train_loss)
    c1_history['train_acc'].append(train_acc)
    c1_history['val_loss'].append(val_loss)
    c1_history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_c1:
        best_val_acc_c1 = val_acc
        
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"Эксперимент C1 завершен. Лучшая accuracy на валидации: {best_val_acc_c1:.4f}")

Запуск эксперимента C1 (simple-cnn-base)...
Epoch 1/5 | Train Loss: 1.8382 Acc: 0.3182 | Val Loss: 1.6328 Acc: 0.3750
Epoch 2/5 | Train Loss: 1.3995 Acc: 0.4825 | Val Loss: 1.3648 Acc: 0.5010
Epoch 3/5 | Train Loss: 1.1723 Acc: 0.5707 | Val Loss: 1.3186 Acc: 0.5040
Epoch 4/5 | Train Loss: 0.9373 Acc: 0.6560 | Val Loss: 1.3044 Acc: 0.5340
Epoch 5/5 | Train Loss: 0.7116 Acc: 0.7432 | Val Loss: 1.4048 Acc: 0.5430
Эксперимент C1 завершен. Лучшая accuracy на валидации: 0.5430


In [5]:
# 1. Трансформации с аугментациями для C2
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(96, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Обновление тренировочного датасета
full_train_dataset_aug = torchvision.datasets.STL10(
    root='./data', 
    split='train', 
    download=False, 
    transform=aug_transform
)

generator = torch.Generator().manual_seed(42)
train_dataset_aug, _ = random_split(full_train_dataset_aug, [train_size, val_size], generator=generator)

aug_train_loader = DataLoader(train_dataset_aug, batch_size=batch_size, shuffle=True)

# 3. Инициализация новой модели с той же архитектурой C1
model_c2 = SimpleCNN().to(device)
optimizer_c2 = optim.Adam(model_c2.parameters(), lr=0.001)

# 4. Запуск цикла обучения C2
c2_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_c2 = 0.0

print("Запуск эксперимента C2 (simple-cnn-aug)...")
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model_c2, aug_train_loader, criterion, optimizer_c2, device)
    # Оценка ВСЕГДА происходит на чистых валидационных данных
    val_loss, val_acc = evaluate(model_c2, val_loader, criterion, device)
    
    c2_history['train_loss'].append(train_loss)
    c2_history['train_acc'].append(train_acc)
    c2_history['val_loss'].append(val_loss)
    c2_history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_c2:
        best_val_acc_c2 = val_acc
        
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"Эксперимент C2 завершен. Лучшая accuracy на валидации: {best_val_acc_c2:.4f}")

Запуск эксперимента C2 (simple-cnn-aug)...
Epoch 1/5 | Train Loss: 1.8726 Acc: 0.3073 | Val Loss: 1.6230 Acc: 0.3730
Epoch 2/5 | Train Loss: 1.5495 Acc: 0.4223 | Val Loss: 1.4085 Acc: 0.4810
Epoch 3/5 | Train Loss: 1.4074 Acc: 0.4850 | Val Loss: 1.3510 Acc: 0.4970
Epoch 4/5 | Train Loss: 1.2889 Acc: 0.5220 | Val Loss: 1.3011 Acc: 0.5220
Epoch 5/5 | Train Loss: 1.1937 Acc: 0.5610 | Val Loss: 1.2272 Acc: 0.5550
Эксперимент C2 завершен. Лучшая accuracy на валидации: 0.5550


In [7]:
from torchvision.models import resnet18, ResNet18_Weights

# 1. Специфические трансформации для ResNet18
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Пересборка датасетов под новые трансформации
full_train_dataset_resnet = torchvision.datasets.STL10(root='./data', split='train', download=False, transform=resnet_transform)
test_dataset_resnet = torchvision.datasets.STL10(root='./data', split='test', download=False, transform=resnet_transform)

# Используем тот же seed
generator = torch.Generator().manual_seed(42)
train_dataset_resnet, val_dataset_resnet = random_split(full_train_dataset_resnet, [train_size, val_size], generator=generator)

resnet_train_loader = DataLoader(train_dataset_resnet, batch_size=batch_size, shuffle=True)
resnet_val_loader = DataLoader(val_dataset_resnet, batch_size=batch_size, shuffle=False)
resnet_test_loader = DataLoader(test_dataset_resnet, batch_size=batch_size, shuffle=False)

# 3. Инициализация предобученной модели
weights = ResNet18_Weights.DEFAULT
model_c3 = resnet18(weights=weights).to(device)

# 4. Заморозка backbone (всех градиентов)
for param in model_c3.parameters():
    param.requires_grad = False

# 5. Замена classification head под 10 классов STL10
# У нового слоя requires_grad автоматически равно True
num_ftrs = model_c3.fc.in_features
model_c3.fc = nn.Linear(num_ftrs, 10).to(device)

# 6. Оптимизатор настраивается ТОЛЬКО на параметры головы
optimizer_c3 = optim.Adam(model_c3.fc.parameters(), lr=0.001)

# 7. Запуск цикла обучения C3
c3_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_c3 = 0.0

print("Запуск эксперимента C3 (resnet18-head-only)...")
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model_c3, resnet_train_loader, criterion, optimizer_c3, device)
    val_loss, val_acc = evaluate(model_c3, resnet_val_loader, criterion, device)
    
    c3_history['train_loss'].append(train_loss)
    c3_history['train_acc'].append(train_acc)
    c3_history['val_loss'].append(val_loss)
    c3_history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_c3:
        best_val_acc_c3 = val_acc
        
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"Эксперимент C3 завершен. Лучшая accuracy на валидации: {best_val_acc_c3:.4f}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Redmi/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100.0%


Запуск эксперимента C3 (resnet18-head-only)...
Epoch 1/5 | Train Loss: 0.9947 Acc: 0.7570 | Val Loss: 0.4384 Acc: 0.9120
Epoch 2/5 | Train Loss: 0.3499 Acc: 0.9275 | Val Loss: 0.2850 Acc: 0.9270
Epoch 3/5 | Train Loss: 0.2492 Acc: 0.9417 | Val Loss: 0.2461 Acc: 0.9380
Epoch 4/5 | Train Loss: 0.2125 Acc: 0.9465 | Val Loss: 0.2200 Acc: 0.9400
Epoch 5/5 | Train Loss: 0.1828 Acc: 0.9525 | Val Loss: 0.2069 Acc: 0.9390
Эксперимент C3 завершен. Лучшая accuracy на валидации: 0.9400


In [8]:
# 1. Инициализация новой модели для C4
model_c4 = resnet18(weights=ResNet18_Weights.DEFAULT).to(device)

# 2. Заморозка всех параметров
for param in model_c4.parameters():
    param.requires_grad = False

# 3. Разморозка layer4 
for param in model_c4.layer4.parameters():
    param.requires_grad = True

# 4. Замена и разморозка классификационной головы
num_ftrs_c4 = model_c4.fc.in_features
model_c4.fc = nn.Linear(num_ftrs_c4, 10).to(device)

# 5. Оптимизатор с дифференцированным learning rate
params_to_update = [
    {'params': model_c4.layer4.parameters(), 'lr': 1e-4},
    {'params': model_c4.fc.parameters(), 'lr': 1e-3}
]
optimizer_c4 = optim.Adam(params_to_update)

# 6. Запуск цикла обучения C4
c4_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_c4 = 0.0

print("Запуск эксперимента C4 (resnet18-finetune)...")
for epoch in range(epochs):
    # Используем resnet_train_loader с нужными трансформациями (224x224)
    train_loss, train_acc = train_one_epoch(model_c4, resnet_train_loader, criterion, optimizer_c4, device)
    val_loss, val_acc = evaluate(model_c4, resnet_val_loader, criterion, device)
    
    c4_history['train_loss'].append(train_loss)
    c4_history['train_acc'].append(train_acc)
    c4_history['val_loss'].append(val_loss)
    c4_history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_c4:
        best_val_acc_c4 = val_acc
        torch.save(model_c4.state_dict(), 'best_classifier.pt')
        
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"Эксперимент C4 завершен. Лучшая accuracy на валидации: {best_val_acc_c4:.4f}")

Запуск эксперимента C4 (resnet18-finetune)...
Epoch 1/5 | Train Loss: 0.4687 Acc: 0.8670 | Val Loss: 0.1781 Acc: 0.9440
Epoch 2/5 | Train Loss: 0.0629 Acc: 0.9898 | Val Loss: 0.1681 Acc: 0.9480
Epoch 3/5 | Train Loss: 0.0194 Acc: 0.9985 | Val Loss: 0.1637 Acc: 0.9480
Epoch 4/5 | Train Loss: 0.0079 Acc: 0.9998 | Val Loss: 0.1670 Acc: 0.9510
Epoch 5/5 | Train Loss: 0.0043 Acc: 1.0000 | Val Loss: 0.1638 Acc: 0.9500
Эксперимент C4 завершен. Лучшая accuracy на валидации: 0.9510


In [9]:
import os
import json
import pandas as pd

os.makedirs('artifacts/figures', exist_ok=True)

# 2. Финальное тестирование лучшей модели (C4)
print("Запуск финального тестирования на test_dataset...")
# Загружаем лучшие веса, которые мы сохранили на последней эпохе C4
model_c4.load_state_dict(torch.load('best_classifier.pt')) 
test_loss, test_acc = evaluate(model_c4, resnet_test_loader, criterion, device)
print(f"Финальная точность на тесте (Test Accuracy): {test_acc:.4f}")

# Перемещаем файл с весами в папку артефактов
os.replace('best_classifier.pt', 'artifacts/best_classifier.pt')

# 3. Сохранение конфигурации
config = {
    "dataset": "STL10",
    "architecture": "ResNet18 (partial finetune layer4+fc)",
    "transforms": "Resize(224), Normalize(ImageNet)",
    "optimizer": "Adam",
    "lr_layer4": 1e-4,
    "lr_fc": 1e-3,
    "epochs": 5,
    "seed": 42,
    "best_val_accuracy": best_val_acc_c4,
    "test_accuracy": test_acc
}
with open('artifacts/best_classifier_config.json', 'w') as f:
    json.dump(config, f, indent=4)

# 4. Формирование runs.csv
runs_data = [
    {'experiment_id': 'C1', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'SimpleCNN', 'optimizer': 'Adam', 'lr': 0.001, 'epochs_trained': 5, 'best_val_accuracy': best_val_acc_c1, 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'Base CNN no augs'},
    {'experiment_id': 'C2', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'SimpleCNN', 'optimizer': 'Adam', 'lr': 0.001, 'epochs_trained': 5, 'best_val_accuracy': best_val_acc_c2, 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'Base CNN with augs'},
    {'experiment_id': 'C3', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'ResNet18_head_only', 'optimizer': 'Adam', 'lr': 0.001, 'epochs_trained': 5, 'best_val_accuracy': best_val_acc_c3, 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'Pretrained backbone frozen'},
    {'experiment_id': 'C4', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'ResNet18_finetune', 'optimizer': 'Adam', 'lr': '1e-4/1e-3', 'epochs_trained': 5, 'best_val_accuracy': best_val_acc_c4, 'test_accuracy': test_acc, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'layer4 and fc finetuned'}
]
df_runs = pd.DataFrame(runs_data)
df_runs.to_csv('artifacts/runs.csv', index=False)

# 5. Генерация графиков
# График кривых обучения лучшей модели
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(c4_history['train_loss'], label='Train Loss')
plt.plot(c4_history['val_loss'], label='Val Loss')
plt.legend()
plt.title('C4: Кривые Loss')

plt.subplot(1, 2, 2)
plt.plot(c4_history['train_acc'], label='Train Acc')
plt.plot(c4_history['val_acc'], label='Val Acc')
plt.legend()
plt.title('C4: Кривые Accuracy')
plt.savefig('artifacts/figures/classification_curves_best.png')
plt.close()

# График сравнения всех экспериментов
plt.figure(figsize=(8, 5))
accs = [best_val_acc_c1, best_val_acc_c2, best_val_acc_c3, best_val_acc_c4]
labels = ['C1 (Base)', 'C2 (Aug)', 'C3 (ResNet Head)', 'C4 (ResNet Fine)']
bars = plt.bar(labels, accs, color=['gray', 'gray', 'steelblue', 'navy'])
plt.ylabel('Validation Accuracy')
plt.title('Сравнение моделей (C1-C4)')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.3f}", ha='center')
plt.savefig('artifacts/figures/classification_compare.png')
plt.close()

# Визуализация аугментаций (берем батч из aug_train_loader)
images, _ = next(iter(aug_train_loader))
grid_img = torchvision.utils.make_grid(images[:16], nrow=4, normalize=True)
plt.figure(figsize=(6, 6))
plt.imshow(grid_img.permute(1, 2, 0).cpu())
plt.axis('off')
plt.title('Примеры аугментаций (C2)')
plt.savefig('artifacts/figures/augmentations_preview.png')
plt.close()

print("Артефакты Части А успешно сгенерированы в директорию artifacts/")

Запуск финального тестирования на test_dataset...
Финальная точность на тесте (Test Accuracy): 0.9543
Артефакты Части А успешно сгенерированы в директорию artifacts/


In [10]:
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
import matplotlib.patches as patches
import xml.etree.ElementTree as ET

print("Инициализация Части B: Детекция (Pascal VOC)...")

# 1. Загрузка предобученной модели детекции
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model_det = fasterrcnn_resnet50_fpn(weights=weights).to(device)
model_det.eval()
print("Модель FasterRCNN загружена.")

# 2. Загрузка датасета Pascal VOC (валидационный сплит)
voc_dataset = torchvision.datasets.VOCDetection(
    root='./data', 
    year='2012', 
    image_set='val', 
    download=True
)
print(f"Датасет VOC загружен. Количество изображений: {len(voc_dataset)}")

# 3. Функция вычисления IoU (Intersection over Union)
def calculate_iou(box1, box2):
    # Координаты пересечения
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    # Площадь пересечения
    intersection_area = max(0, x2 - x1) * max(0, y2 - y1)

    # Площади обоих боксов
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # Площадь объединения
    union_area = box1_area + box2_area - intersection_area

    # Защита от деления на ноль
    if union_area == 0:
        return 0.0

    return intersection_area / union_area

# 4. Функция парсинга Ground Truth из VOC
def get_voc_ground_truth(annotation):
    gt_boxes = []
    objects = annotation['annotation'].get('object', [])
    if not isinstance(objects, list):
        objects = [objects]
        
    for obj in objects:
        bndbox = obj['bndbox']
        xmin = float(bndbox['xmin'])
        ymin = float(bndbox['ymin'])
        xmax = float(bndbox['xmax'])
        ymax = float(bndbox['ymax'])
        gt_boxes.append([xmin, ymin, xmax, ymax])
    return gt_boxes

print("Математический аппарат для расчета IoU подготовлен.")

Инициализация Части B: Детекция (Pascal VOC)...


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\Redmi/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100.0%


Модель FasterRCNN загружена.


100.0%


Датасет VOC загружен. Количество изображений: 5823
Математический аппарат для расчета IoU подготовлен.


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

# Количество изображений для базовой оценки 
num_eval_images = 50

def evaluate_detection(model, dataset, threshold, num_images, device):
    tps = 0
    fps = 0
    fns = 0
    iou_list = []
    
    for i in range(num_images):
        img, annotation = dataset[i]
        gt_boxes = get_voc_ground_truth(annotation)
        
        img_tensor = TF.to_tensor(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            prediction = model(img_tensor)[0]
            
        # Фильтрация по порогу уверенности
        scores = prediction['scores'].cpu().numpy()
        boxes = prediction['boxes'].cpu().numpy()
        
        keep_idx = scores >= threshold
        pred_boxes = boxes[keep_idx]
        
        # Сопоставление предсказаний с Ground Truth (IoU >= 0.5)
        matched_gt = set()
        for p_box in pred_boxes:
            best_iou = 0
            best_gt_idx = -1
            for j, g_box in enumerate(gt_boxes):
                if j in matched_gt:
                    continue
                iou = calculate_iou(p_box, g_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = j
            
            if best_iou >= 0.5:
                tps += 1
                matched_gt.add(best_gt_idx)
                iou_list.append(best_iou)
            else:
                fps += 1
                
        fns += len(gt_boxes) - len(matched_gt)
        
    precision = tps / (tps + fps) if (tps + fps) > 0 else 0
    recall = tps / (tps + fns) if (tps + fns) > 0 else 0
    mean_iou = np.mean(iou_list) if iou_list else 0
    
    return precision, recall, mean_iou

print("Инференс режима V1 (score_threshold = 0.3)...")
p_v1, r_v1, iou_v1 = evaluate_detection(model_det, voc_dataset, 0.3, num_eval_images, device)
print(f"V1 -> Precision: {p_v1:.4f}, Recall: {r_v1:.4f}, Mean IoU: {iou_v1:.4f}")

print("Инференс режима V2 (score_threshold = 0.7)...")
p_v2, r_v2, iou_v2 = evaluate_detection(model_det, voc_dataset, 0.7, num_eval_images, device)
print(f"V2 -> Precision: {p_v2:.4f}, Recall: {r_v2:.4f}, Mean IoU: {iou_v2:.4f}")

img_vis, ann_vis = voc_dataset[0]
gt_boxes_vis = get_voc_ground_truth(ann_vis)
img_tensor_vis = TF.to_tensor(img_vis).unsqueeze(0).to(device)

with torch.no_grad():
    pred_vis = model_det(img_tensor_vis)[0]
scores_vis = pred_vis['scores'].cpu().numpy()
boxes_vis = pred_vis['boxes'].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, thresh, title in zip(axes, [0.3, 0.7], ['V1: Threshold 0.3', 'V2: Threshold 0.7']):
    ax.imshow(img_vis)
    ax.axis('off')
    ax.set_title(title)
    
    # Отрисовка Ground Truth (зеленые)
    for g_box in gt_boxes_vis:
        rect = patches.Rectangle((g_box[0], g_box[1]), g_box[2]-g_box[0], g_box[3]-g_box[1], linewidth=2, edgecolor='g', facecolor='none', label='GT')
        ax.add_patch(rect)
        
    # Отрисовка предсказаний (красные)
    keep_idx = scores_vis >= thresh
    for p_box in boxes_vis[keep_idx]:
        rect = patches.Rectangle((p_box[0], p_box[1]), p_box[2]-p_box[0], p_box[3]-p_box[1], linewidth=2, edgecolor='r', facecolor='none', label='Pred')
        ax.add_patch(rect)

plt.savefig('artifacts/figures/detection_examples.png')
plt.close()

# График метрик
metrics_df = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'Mean IoU'],
    'V1 (Thr 0.3)': [p_v1, r_v1, iou_v1],
    'V2 (Thr 0.7)': [p_v2, r_v2, iou_v2]
})
metrics_df.plot(x='Metric', kind='bar', figsize=(8, 5))
plt.title('Сравнение метрик детекции: V1 vs V2')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.savefig('artifacts/figures/detection_metrics.png')
plt.close()

# Обновление runs.csv
df_runs = pd.read_csv('artifacts/runs.csv')
new_runs = pd.DataFrame([
    {'experiment_id': 'V1', 'task': 'detection', 'dataset': 'VOCDetection', 'seed': 42, 'model_summary': 'FasterRCNN_ResNet50_FPN', 'optimizer': None, 'lr': None, 'epochs_trained': None, 'best_val_accuracy': None, 'test_accuracy': None, 'precision': p_v1, 'recall': r_v1, 'mean_iou': iou_v1, 'notes': 'threshold 0.3'},
    {'experiment_id': 'V2', 'task': 'detection', 'dataset': 'VOCDetection', 'seed': 42, 'model_summary': 'FasterRCNN_ResNet50_FPN', 'optimizer': None, 'lr': None, 'epochs_trained': None, 'best_val_accuracy': None, 'test_accuracy': None, 'precision': p_v2, 'recall': r_v2, 'mean_iou': iou_v2, 'notes': 'threshold 0.7'}
])
df_runs = pd.concat([df_runs, new_runs], ignore_index=True)
df_runs.to_csv('artifacts/runs.csv', index=False)

print("Артефакты Части B успешно сгенерированы. Проверьте папку artifacts/figures/")

Инференс режима V1 (score_threshold = 0.3)...
V1 -> Precision: 0.2784, Recall: 0.9450, Mean IoU: 0.8041
Инференс режима V2 (score_threshold = 0.7)...
V2 -> Precision: 0.5488, Recall: 0.8257, Mean IoU: 0.8171
Артефакты Части B успешно сгенерированы. Проверьте папку artifacts/figures/


C:\Users\Redmi\AppData\Local\Temp\ipykernel_29072\3123810300.py:117: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_runs = pd.concat([df_runs, new_runs], ignore_index=True)
